# Mule Validation — does behavioural distance surface known mules?

**What this does:** takes a list of confirmed-mule ACIDs and reports where each one
ranks in the behavioural-distance ordering against your watchlist reference accounts.
It reuses the *exact* production feature/scaling/distance code, so results match the
live pipeline.

**How to use:** edit `CONFIRMED_MULES` in the config cell, run all cells, and paste the
output of the final cell (**PASTE THIS BACK**) into the chat. The notebook loads `.env`
itself via python-dotenv, so just launch Jupyter from the project folder — no need to
source `.env` first. (Requires `pip install python-dotenv`.)

**Honest caveat at small n:** with only a handful of labels this is a *directional*
check, not a verdict. The same notebook becomes conclusive as the list grows — no
rework needed.

## 1. Config — the only cell you normally edit

In [1]:
# ---- Confirmed mule accounts (edit this list; add more as you validate) ----
CONFIRMED_MULES = [
    "UG6119095",   # <-- placeholder, replace with a real confirmed ACID
    "UG6119094",   # <-- placeholder
    "UG6119032",   # <-- placeholder
    "UG6052746",   # <-- placeholder   # <-- placeholder
    "UG6112062",   # <-- placeholder
    "UG6094534",
    "UG6154756",
    "UG6310606",
    # add more here, one ACID per line
]

# ---- Reference accounts to rank the mules against ----
# Leave as None to auto-load the live watchlist from the reporting DB,
# or hard-code a list of reference ACIDs, e.g. ["UG6246002", "UG6370211"].
REFERENCE_ACIDS = ["UG6246002", "UG6370211"]

# ---- Data source ----
# True  = rebuild profiles fresh from Finacle (~4 min, always current)
# False = load a cached profiles parquet (instant) if you've saved one
REBUILD_PROFILES = False
PROFILES_CACHE = "profiles_cache.parquet"   # used only when REBUILD_PROFILES = False

## 2. Build (or load) profiles using the production code

In [2]:
import os
import numpy as np
import pandas as pd

# Load the AML_* connection variables from .env directly (no need to source
# .env in the shell first). Values already set in the environment win, so a
# sourced .env or exported vars are respected; override=False keeps them.
from dotenv import load_dotenv
load_dotenv(dotenv_path='.env', override=False)
assert os.environ.get('AML_READ_USER') or os.environ.get('AML_READ_URL'), \
    'AML_READ_* not found — check that .env exists in this folder and is filled in'

import aml_screening as m

if REBUILD_PROFILES:
    raw = m.extract_transactions_db()
    txns = m.clean_transactions(raw)
    profiles = m.build_profiles(txns)
    profiles.to_parquet(PROFILES_CACHE)   # save so you can iterate fast next time
    print(f"Rebuilt and cached {len(profiles):,} account profiles")
else:
    profiles = pd.read_parquet(PROFILES_CACHE)
    print(f"Loaded {len(profiles):,} cached account profiles")

profiles = profiles.reset_index(drop=True)

Loaded 24,095 cached account profiles


## 3. Scope check — how many listed mules are actually in the book?

In [3]:
in_scope = [a for a in CONFIRMED_MULES if a in set(profiles['ACID'])]
missing  = [a for a in CONFIRMED_MULES if a not in set(profiles['ACID'])]

print(f"Listed mules      : {len(CONFIRMED_MULES)}")
print(f"In profiled book  : {len(in_scope)}")
print(f"Out of scope      : {len(missing)}  (not USD / not opened >=2024 / no txns)")
if missing:
    print("  missing ACIDs:", missing)

# resolve reference accounts
if REFERENCE_ACIDS is None:
    write_eng = m._engine('WRITE')
    ref_acids = m.load_watchlist_db(write_eng)
    print(f"\nReference accounts (from live watchlist): {ref_acids}")
else:
    ref_acids = REFERENCE_ACIDS
    print(f"\nReference accounts (hard-coded): {ref_acids}")

Listed mules      : 8
In profiled book  : 8
Out of scope      : 0  (not USD / not opened >=2024 / no txns)

Reference accounts (hard-coded): ['UG6246002', 'UG6370211']


## 4. Rank each mule against each reference account

Uses the production scaler + distance. For each reference account we compute every
account's distance to it, then find each mule's **rank** and **percentile** in that
full ordering (percentile = fraction of the book at least as close; lower = closer).

In [4]:
# scale once with the production RobustScaler + clip + median-impute.
# NOTE: mules that are themselves on the watchlist are NOT ranked against
# themselves (self-match = rank 1, distance 0), so ranks are honest.
X = m.scale_features(profiles)
N = len(profiles)
acid_to_row = {a: i for i, a in enumerate(profiles['ACID'])}
name_of = dict(zip(profiles['ACID'], profiles['ACCT_NAME']))

records = []
for ref in ref_acids:
    if ref not in acid_to_row:
        print(f"(reference {ref} not in book — skipped)")
        continue
    ref_i = acid_to_row[ref]
    dist = np.linalg.norm(X - X[ref_i], axis=1)
    order = np.argsort(dist)                       # closest first
    rank_of_row = {row: r for r, row in enumerate(order, start=1)}
    for a in in_scope:
        if a == ref:
            continue                       # v2.6 fix: skip self-match (mule is this seed)
        i = acid_to_row[a]
        rank = rank_of_row[i]
        records.append({
            'mule_acid': a,
            'mule_name': name_of[a],
            'reference': name_of.get(ref, ref),
            'distance': round(float(dist[i]), 3),
            'rank': rank,
            'percentile': round(100 * rank / N, 3),   # e.g. 0.4 = top 0.4%
        })

ranks = pd.DataFrame(records).sort_values(['reference', 'rank'])
ranks

,mule_acid,mule_name,reference,distance,rank,percentile
7,UG6310606,OMOSPIRE UGANDA LIMITED,FRONTIER TEXTILE DEALERS CO LIMITED,1.226,708,2.938
2,UG6119032,KWELIVEST INVESTMENT SMC LIMITED,FRONTIER TEXTILE DEALERS CO LIMITED,1.801,5329,22.117
1,UG6119094,AETHERGRID TECHNOLOGIES LIMITED,FRONTIER TEXTILE DEALERS CO LIMITED,1.903,6319,26.225
0,UG6119095,NEBULACORE INNOVATIONS LIMITED,FRONTIER TEXTILE DEALERS CO LIMITED,1.979,6970,28.927
3,UG6052746,PACIFIC PRINCE INTERNATIONAL (U) CO. LIMITED,FRONTIER TEXTILE DEALERS CO LIMITED,2.414,11283,46.827
4,UG6112062,EMIC VENTURES,FRONTIER TEXTILE DEALERS CO LIMITED,2.477,11775,48.869
5,UG6094534,HESHIMA PRIME LOGISTICS LTD,FRONTIER TEXTILE DEALERS CO LIMITED,2.628,12937,53.692
6,UG6154756,SHAUPAM ENTERPRISES LIMITED,FRONTIER TEXTILE DEALERS CO LIMITED,2.785,13996,58.087
15,UG6310606,OMOSPIRE UGANDA LIMITED,RAVO TECH COMPANY SMC LIMITED,0.742,22,0.091
10,UG6119032,KWELIVEST INVESTMENT SMC LIMITED,RAVO TECH COMPANY SMC LIMITED,1.085,128,0.531


## 5. Each mule's *best* match across all references

In [5]:
best = (ranks.sort_values('rank')
             .groupby('mule_acid', as_index=False)
             .first()[['mule_acid', 'mule_name', 'reference', 'distance', 'rank', 'percentile']])
best = best.sort_values('rank')
best

,mule_acid,mule_name,reference,distance,rank,percentile
7,UG6310606,OMOSPIRE UGANDA LIMITED,RAVO TECH COMPANY SMC LIMITED,0.742,22,0.091
3,UG6119032,KWELIVEST INVESTMENT SMC LIMITED,RAVO TECH COMPANY SMC LIMITED,1.085,128,0.531
4,UG6119094,AETHERGRID TECHNOLOGIES LIMITED,RAVO TECH COMPANY SMC LIMITED,1.085,129,0.535
5,UG6119095,NEBULACORE INNOVATIONS LIMITED,RAVO TECH COMPANY SMC LIMITED,1.122,148,0.614
0,UG6052746,PACIFIC PRINCE INTERNATIONAL (U) CO. LIMITED,RAVO TECH COMPANY SMC LIMITED,1.666,692,2.872
2,UG6112062,EMIC VENTURES,RAVO TECH COMPANY SMC LIMITED,1.828,1157,4.802
1,UG6094534,HESHIMA PRIME LOGISTICS LTD,RAVO TECH COMPANY SMC LIMITED,1.926,1534,6.366
6,UG6154756,SHAUPAM ENTERPRISES LIMITED,RAVO TECH COMPANY SMC LIMITED,1.932,1555,6.454


## 6. Descriptive feature comparison — mules vs the book (hypothesis only)

In [6]:
fp_cols = ['flow_through_ratio', 'largest_credit_share', 'largest_debit_share',
           'top3_credit_share', 'top3_debit_share', 'median_hours_to_next_debit',
           'frac_credits_followed_24h', 'account_age_days', 'turnover_per_day']

mule_rows = profiles[profiles['ACID'].isin(in_scope)]
compare = pd.DataFrame({
    'mule_median': mule_rows[fp_cols].median(),
    'book_median': profiles[fp_cols].median(),
}).round(3)
compare['mule_vs_book'] = (compare['mule_median'] / compare['book_median']).round(2)
compare

,mule_median,book_median,mule_vs_book
flow_through_ratio,0.999,0.983,1.02
largest_credit_share,0.074,0.600,0.12
largest_debit_share,0.054,0.476,0.11
top3_credit_share,0.186,0.995,0.19
top3_debit_share,0.130,0.857,0.15
median_hours_to_next_debit,2.621,23.371,0.11
frac_credits_followed_24h,0.916,0.500,1.83
account_age_days,357.500,228.000,1.57
turnover_per_day,42279.451,67.125,629.86


## 7. PASTE THIS BACK — copy the whole output of this cell into the chat

In [7]:
print("=" * 60)
print("MULE VALIDATION SUMMARY")
print("=" * 60)
print(f"Profiled accounts in book : {N:,}")
print(f"Mules listed / in scope   : {len(CONFIRMED_MULES)} / {len(in_scope)}")
print(f"Reference accounts used   : {len(ranks['reference'].unique()) if len(ranks) else 0}")
print()
print("Best rank per mule (across all references):")
if len(ranks):
    for _, r in best.iterrows():
        print(f"  rank {int(r['rank']):>6}  (top {r['percentile']:.2f}%)  "
              f"dist {r['distance']:.3f}  {r['mule_name'][:32]:<32}  <- {r['reference'][:24]}")
    print()
    print(f"Best-rank percentile  -> median {best['percentile'].median():.2f}%  "
          f"| range {best['percentile'].min():.2f}%-{best['percentile'].max():.2f}%")
    print(f"How many mules land in top 1%  : {(best['percentile'] <= 1).sum()} / {len(best)}")
    print(f"How many mules land in top 5%  : {(best['percentile'] <= 5).sum()} / {len(best)}")
else:
    print("  (no in-scope mules to rank — check the ACID list and scope output)")
print()
print("Feature medians (mule vs book, ratio):")
for f in fp_cols:
    print(f"  {f:<28} mule {compare.loc[f,'mule_median']:>10}  "
          f"book {compare.loc[f,'book_median']:>10}  x{compare.loc[f,'mule_vs_book']}")
print("=" * 60)

MULE VALIDATION SUMMARY
Profiled accounts in book : 24,095
Mules listed / in scope   : 8 / 8
Reference accounts used   : 2

Best rank per mule (across all references):
  rank     22  (top 0.09%)  dist 0.742  OMOSPIRE UGANDA LIMITED           <- RAVO TECH COMPANY SMC LI
  rank    128  (top 0.53%)  dist 1.085  KWELIVEST INVESTMENT SMC LIMITED  <- RAVO TECH COMPANY SMC LI
  rank    129  (top 0.54%)  dist 1.085  AETHERGRID TECHNOLOGIES LIMITED   <- RAVO TECH COMPANY SMC LI
  rank    148  (top 0.61%)  dist 1.122  NEBULACORE INNOVATIONS LIMITED    <- RAVO TECH COMPANY SMC LI
  rank    692  (top 2.87%)  dist 1.666  PACIFIC PRINCE INTERNATIONAL (U)  <- RAVO TECH COMPANY SMC LI
  rank   1157  (top 4.80%)  dist 1.828  EMIC VENTURES                     <- RAVO TECH COMPANY SMC LI
  rank   1534  (top 6.37%)  dist 1.926  HESHIMA PRIME LOGISTICS LTD       <- RAVO TECH COMPANY SMC LI
  rank   1555  (top 6.45%)  dist 1.932  SHAUPAM ENTERPRISES LIMITED       <- RAVO TECH COMPANY SMC LI

Best-rank perce

## 8. Experiment — does up-weighting velocity tighten the mule ranking?

The feature comparison suggested velocity (`median_hours_to_next_debit`,
`frac_credits_followed_24h`) separates mules from the book far more than the other
features. Here we multiply those two columns in the scaled space by a weight and
re-rank, to see whether the mules' percentile improves. **Self-matches are excluded**
(a mule that is itself a reference account is not scored against itself), so this is a
held-out test. Weight = 1.0 is the current production distance.

*Read with care at small n:* a lower median percentile here is encouraging, but we
only fold a weight into production if it helps AND holds as more labels are added.

In [8]:
VELOCITY_COLS = ['median_hours_to_next_debit', 'frac_credits_followed_24h']
WEIGHTS_TO_TRY = [1.0, 1.5, 2.0, 3.0, 5.0, 8.0]

vel_idx = [m.BEHAVIOUR_COLS.index(c) for c in VELOCITY_COLS]
ref_set = set(ref_acids)

def held_out_percentiles(weight):
    """Best percentile per mule across references, excluding self-matches."""
    Xw = X.copy()
    Xw[:, vel_idx] *= weight
    best_pct = {}
    for ref in ref_acids:
        if ref not in acid_to_row:
            continue
        ri = acid_to_row[ref]
        dist = np.linalg.norm(Xw - Xw[ri], axis=1)
        order = np.argsort(dist)
        rank_of = {row: r for r, row in enumerate(order, start=1)}
        for a in in_scope:
            if a == ref:
                continue                      # skip self-match
            pct = 100 * rank_of[acid_to_row[a]] / N
            if a not in best_pct or pct < best_pct[a]:
                best_pct[a] = pct
    return best_pct

sweep = []
for w in WEIGHTS_TO_TRY:
    bp = held_out_percentiles(w)
    vals = np.array(list(bp.values()))
    sweep.append({
        'weight': w,
        'median_pct': round(float(np.median(vals)), 3),
        'max_pct': round(float(vals.max()), 3),
        'in_top_1pct': int((vals <= 1).sum()),
        'in_top_5pct': int((vals <= 5).sum()),
        'n_mules': len(vals),
    })
sweep_df = pd.DataFrame(sweep)
sweep_df

,weight,median_pct,max_pct,in_top_1pct,in_top_5pct,n_mules
0,1.0,1.743,6.454,4,6,8
1,1.5,1.662,6.217,4,6,8
2,2.0,1.646,6.043,4,6,8
3,3.0,1.670,6.275,4,6,8
4,5.0,2.762,8.209,3,5,8
5,8.0,7.966,13.053,1,3,8


In [9]:
# Per-mule: baseline (w=1) vs a chosen weight, side by side
COMPARE_WEIGHT = 3.0        # change to whichever weight the sweep favours

base_bp = held_out_percentiles(1.0)
wt_bp   = held_out_percentiles(COMPARE_WEIGHT)
side = pd.DataFrame({
    'mule': [name_of[a][:32] for a in base_bp],
    'baseline_pct': [round(base_bp[a], 3) for a in base_bp],
    f'weighted_x{COMPARE_WEIGHT}_pct': [round(wt_bp[a], 3) for a in base_bp],
}).sort_values('baseline_pct')
side['improved'] = side[f'weighted_x{COMPARE_WEIGHT}_pct'] < side['baseline_pct']
side

,mule,baseline_pct,weighted_x3.0_pct,improved
7,OMOSPIRE UGANDA LIMITED,0.091,0.390,False
2,KWELIVEST INVESTMENT SMC LIMITED,0.531,0.245,True
1,AETHERGRID TECHNOLOGIES LIMITED,0.535,0.436,True
0,NEBULACORE INNOVATIONS LIMITED,0.614,0.519,True
3,PACIFIC PRINCE INTERNATIONAL (U),2.872,2.822,True
4,EMIC VENTURES,4.802,4.615,True
5,HESHIMA PRIME LOGISTICS LTD,6.366,5.624,True
6,SHAUPAM ENTERPRISES LIMITED,6.454,6.275,True


## 9. PASTE THIS BACK (experiment) — copy the whole output of this cell

In [10]:
print("=" * 60)
print("VELOCITY-WEIGHT EXPERIMENT")
print("=" * 60)
print(f"Held-out mules scored : {sweep[0]['n_mules']}  (self-matches excluded)")
print(f"Velocity columns      : {VELOCITY_COLS}")
print()
print(f"{'weight':>7} {'median%':>9} {'max%':>8} {'top1%':>7} {'top5%':>7}")
for s in sweep:
    print(f"{s['weight']:>7} {s['median_pct']:>9} {s['max_pct']:>8} "
          f"{s['in_top_1pct']:>4}/{s['n_mules']:<2} {s['in_top_5pct']:>4}/{s['n_mules']:<2}")
print()
base_med = sweep[0]['median_pct']
best_row = min(sweep, key=lambda s: s['median_pct'])
print(f"Baseline (w=1) median : {base_med}%")
print(f"Best weight           : {best_row['weight']}  -> median {best_row['median_pct']}%")
print("=" * 60)

VELOCITY-WEIGHT EXPERIMENT
Held-out mules scored : 8  (self-matches excluded)
Velocity columns      : ['median_hours_to_next_debit', 'frac_credits_followed_24h']

 weight   median%     max%   top1%   top5%
    1.0     1.743    6.454    4/8     6/8 
    1.5     1.662    6.217    4/8     6/8 
    2.0     1.646    6.043    4/8     6/8 
    3.0      1.67    6.275    4/8     6/8 
    5.0     2.762    8.209    3/8     5/8 
    8.0     7.966   13.053    1/8     3/8 

Baseline (w=1) median : 1.743%
Best weight           : 2.0  -> median 1.646%
